In [13]:
import os
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(torch.cuda.get_device_capability(0))
    print(torch.cuda.get_device_properties(0))

device = torch.device('cuda' if torch.cuda.is_available else 'cpu')
print(f" Executing production pipeline on device: {device}")
    

NVIDIA GeForce RTX 2050
(8, 6)
_CudaDeviceProperties(name='NVIDIA GeForce RTX 2050', major=8, minor=6, total_memory=4095MB, multi_processor_count=16, uuid=67ed86ee-3100-a949-9ad2-6e8bb533bd81, L2_cache_size=1MB)
 Executing production pipeline on device: cuda


In [19]:
transfrom = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((.1307),(.3081))
])

train_dataset = datasets.MNIST(root='./data',train=True,download=True,transform=transfrom)
test_dataset = datasets.MNIST(root='./data',train=False,download=True,transform=transfrom)

BATCH_SIZE=128
NUM_CLASSES = 10
INPUT_FEATURE =  28*28

train_loader = DataLoader(dataset=train_dataset,shuffle=True,batch_size=BATCH_SIZE,drop_last=True)
test_loader = DataLoader(dataset=train_dataset,shuffle=False,batch_size=BATCH_SIZE)

print(f"Data Batches Configured: {len(train_loader)} training steps per epoch.")


Data Batches Configured: 468 training steps per epoch.


In [20]:

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(in_features=INPUT_FEATURE,out_features=512),
    nn.ReLU(),
    nn.Dropout(p=.2),
    nn.Linear(in_features=512,out_features=256),
    nn.ReLU(),
    nn.Dropout(p=.2),
    nn.Linear(in_features=256,out_features=NUM_CLASSES)
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=512, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.2, inplace=False)
  (4): Linear(in_features=512, out_features=256, bias=True)
  (5): ReLU()
  (6): Dropout(p=0.2, inplace=False)
  (7): Linear(in_features=256, out_features=10, bias=True)
)


In [22]:
def evaluate_accuracy(loader, model_engine):
    model_engine.eval()  # Turn off dropout behaviors during evaluation
    correct_predictions, total_samples = 0, 0
    
    with torch.no_grad():  # Turn off autograd memory tracking to speed up performance
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_engine(images)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()
            
    return (correct_predictions / total_samples) * 100

In [ ]:
EPOCHS = 5
print("Launching Production Image Classifier Training Loop...\n")

for epoch in range(EPOCHS):
    model.train() 
    running_loss = 0.0
    start_time = time.time()
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad(set_to_none=True) # Highly optimized memory handling step
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        
    # Run historical telemetry reviews at the close of every Epoch step
    epoch_duration = time.time() - start_time
    average_batch_loss = running_loss / len(train_loader)
    test_accuracy = evaluate_accuracy(test_loader, model)
    
    print(f" Epoch [{epoch+1}/{EPOCHS}] | Duration: {epoch_duration:.2f}s")
    print(f"   ↳ Training Step Batch Loss: {average_batch_loss:.4f}")
    print(f"   ↳ Evaluation Dataset Accuracy: {test_accuracy:.2f}%\n")

checkpoint_path = "production_mlp_classifier.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.step,
}, checkpoint_path)

print(f" Checkpoint compiled successfully and exported to: '{os.path.abspath(checkpoint_path)}'")

Launching Production Image Classifier Training Loop...

 Epoch [1/5] | Duration: 23.40s
   ↳ Training Step Batch Loss: 0.0737
   ↳ Evaluation Dataset Accuracy: 98.73%

 Epoch [2/5] | Duration: 23.09s
   ↳ Training Step Batch Loss: 0.0606
   ↳ Evaluation Dataset Accuracy: 99.14%

 Epoch [3/5] | Duration: 25.23s
   ↳ Training Step Batch Loss: 0.0480
   ↳ Evaluation Dataset Accuracy: 99.02%

 Epoch [4/5] | Duration: 32.83s
   ↳ Training Step Batch Loss: 0.0459
   ↳ Evaluation Dataset Accuracy: 99.35%

 Epoch [5/5] | Duration: 28.27s
   ↳ Training Step Batch Loss: 0.0393
   ↳ Evaluation Dataset Accuracy: 99.46%



💾 Checkpoint compiled successfully and exported to: 'c:\Users\Ayush\Git Repo\Deep dive into AI\Deep Learning Production Class\production_mlp_classifier.pt'
